# Task 1 Classifier Entrypoint

This notebook is a thin grading wrapper. It reads `data/test_cls.json` and writes `outputs/cls_output.json`.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Colab에서 자동 탐색이 실패하면 Google Drive의 프로젝트 상대 경로를 직접 입력하세요.
# 예: PROJECT_RELATIVE_TO_DRIVE = "Termproject_신해솔"
PROJECT_RELATIVE_TO_DRIVE = ""


def is_project_root(path: Path) -> bool:
    return (path / "chatbot.sh").is_file() and (path / "src").is_dir()


def add_if_valid(path: Path | str | None, tried: list[str], label: str) -> Path | None:
    if not path:
        tried.append(f"{label}: 값 없음")
        return None
    candidate = Path(path).expanduser().resolve()
    tried.append(f"{label}: {candidate}")
    return candidate if is_project_root(candidate) else None


def find_upward(start: Path, tried: list[str]) -> Path | None:
    for candidate in [start.resolve(), *start.resolve().parents]:
        tried.append(f"상위 폴더 탐색: {candidate}")
        if is_project_root(candidate):
            return candidate
    return None


def mount_drive_if_colab(tried: list[str]) -> Path | None:
    if not Path("/content").exists():
        tried.append("Colab Drive 탐색: /content 없음")
        return None
    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.exists():
        try:
            from google.colab import drive  # type: ignore

            drive.mount("/content/drive")
        except Exception as exc:  # noqa: BLE001
            tried.append(f"Colab Drive 마운트 실패: {exc}")
            return None
    if drive_root.exists():
        tried.append(f"Colab Drive 마운트 확인: {drive_root}")
        return drive_root
    tried.append("Colab Drive 탐색: MyDrive 없음")
    return None


def drive_candidates(my_drive: Path):
    patterns = ["Termproject*", "nlp-term"]
    prefixes = ["", "*", "*/*", "*/*/*"]
    for prefix in prefixes:
        for pattern in patterns:
            glob_pattern = f"{prefix}/{pattern}" if prefix else pattern
            yield from my_drive.glob(glob_pattern)


def find_project_root() -> Path:
    tried: list[str] = []

    root = add_if_valid(os.environ.get("NLP_TERM_PROJECT_ROOT"), tried, "NLP_TERM_PROJECT_ROOT")
    if root:
        return root

    root = find_upward(Path.cwd(), tried)
    if root:
        return root

    my_drive = mount_drive_if_colab(tried)
    if my_drive:
        for candidate in drive_candidates(my_drive):
            tried.append(f"Drive 자동 탐색: {candidate}")
            if is_project_root(candidate):
                return candidate.resolve()

        manual_path = my_drive / PROJECT_RELATIVE_TO_DRIVE if PROJECT_RELATIVE_TO_DRIVE else None
        root = add_if_valid(manual_path, tried, "PROJECT_RELATIVE_TO_DRIVE")
        if root:
            return root

    tried_text = "\n".join(f"- {item}" for item in tried)
    raise RuntimeError(
        "프로젝트 루트를 찾지 못했습니다. notebook 상단의 "
        "PROJECT_RELATIVE_TO_DRIVE 값을 수정하거나 NLP_TERM_PROJECT_ROOT 환경변수를 설정하세요.\n"
        f"시도한 경로:\n{tried_text}"
    )


def classifier_dependencies_available() -> bool:
    try:
        import joblib  # noqa: F401
        import pydantic  # noqa: F401
        import sklearn  # noqa: F401
    except ImportError:
        return False
    return True


def ensure_classifier_dependencies() -> None:
    if classifier_dependencies_available():
        return

    try:
        get_ipython().run_line_magic("pip", "install -q scikit-learn joblib pydantic")  # type: ignore[name-defined]
    except NameError:
        pass

    if not classifier_dependencies_available():
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib", "pydantic"])
        except subprocess.CalledProcessError:
            subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib", "pydantic"])

    if not classifier_dependencies_available():
        raise RuntimeError("분류기 의존성 설치에 실패했습니다: scikit-learn, joblib, pydantic")


repo_root = find_project_root()
ensure_classifier_dependencies()

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from nlp_term.classify.predict import predict_file  # noqa: E402
from nlp_term.paths import default_input_path, default_output_path  # noqa: E402

input_path = default_input_path("test_cls.json")
output_path = default_output_path("cls_output.json")
predict_file(input_path, output_path)

rows = json.loads(output_path.read_text(encoding="utf-8"))
print(f"wrote {output_path}")
print(f"processed {len(rows)} rows")


wrote C:\Users\haesol\dev\nlp-term\outputs\cls_output.json
